# Notebook 5: Transforming and Saving Data (Solutions)
**Time: 60-90 minutes**

### What you'll learn
- Process API responses into clean data using loops
- Convert data to pandas DataFrames
- Filter, sort, and select columns
- Export to CSV and Excel
- Save raw JSON for later use

---
## 5.1 Why transform?

APIs give you JSON. Your colleagues want spreadsheets. This notebook bridges that gap.

We'll go from raw API response to a clean CSV/Excel file, step by step.

---
## 5.2 Processing API responses with loops

Let's start with what you already know — loops. We'll fetch European countries and clean the data.

In [ ]:
import requests

response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,capital,population"}
)
countries = response.json()

print(f"Got {len(countries)} countries")
print(f"\nFirst item looks like: {countries[0]}")

In [ ]:
# The data is nested — let's flatten it into simple rows
rows = []
for c in countries:
    rows.append({
        "name": c["name"]["common"],
        "capital": c["capital"][0] if c.get("capital") else "N/A",
        "population": c["population"]
    })

# Print the first 5
for row in rows[:5]:
    print(f"{row['name']}: {row['capital']} ({row['population']})")

This works, but printing isn't great for large datasets. Let's meet a better tool.

---
## 5.3 Introducing pandas: the better way

`pandas` turns a list of dictionaries into a nice table called a **DataFrame**. Each dictionary becomes a row, each key becomes a column.

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df

One line, and you get a clean table! Let's explore what you can do with it.

In [ ]:
# Quick overview
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
df.head(10)  # Show first 10 rows

---
## 5.4 Handling nested API responses

Real APIs nest their data in two common ways, and each needs a different move:

1. **The list is wrapped in a key.** The response is a dictionary, and the rows you want live under a key like `"results"` or `"data"`. You extract that list *first*, then hand it to pandas.
2. **Each item is deeply nested.** The fields you want are buried a few layers down (like `types[0]["type"]["name"]`). You flatten them into flat rows with a loop *before* making the DataFrame.

Let's see both on a real API.

In [ ]:
# Pattern 1: the list is wrapped in a key.
# PokéAPI returns a dictionary — the rows live under "results".
response = requests.get(
    "https://pokeapi.co/api/v2/pokemon",
    params={"limit": 5}
)
response_data = response.json()

print("Top-level keys:", list(response_data.keys()))

# Extract the list FIRST, then make a DataFrame
df_pokemon = pd.DataFrame(response_data["results"])
df_pokemon

Pattern 2: when each item is deeply nested, a column you want can be buried several layers down. Flatten the parts you care about with a loop first:

In [ ]:
# Each Pokémon's type is nested at types[0]["type"]["name"]
pikachu = requests.get("https://pokeapi.co/api/v2/pokemon/pikachu").json()
print("Raw type field:", pikachu["types"][0])

# Flatten the parts we want into simple rows
names = ["pikachu", "charizard", "bulbasaur"]
rows = []
for name in names:
    p = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}").json()
    rows.append({
        "name": p["name"],
        "height": p["height"],
        "weight": p["weight"],
        "main_type": p["types"][0]["type"]["name"]  # dig down to the name
    })

df_flat = pd.DataFrame(rows)
df_flat

**Exercise:** Fetch European countries with fields `name,capital,population,area`. Flatten each one with a loop and build a DataFrame.

Required variables:
- `df_exercise` — a pandas DataFrame with columns `name`, `capital`, `population`, and `area_km2` (one row per country)

In [ ]:
# Fetch European countries with name, capital, population, and area
response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,capital,population,area"}
)
countries = response.json()

# Flatten with a loop
rows = []
for c in countries:
    rows.append({
        "name": c["name"]["common"],
        "capital": c["capital"][0] if c.get("capital") else "N/A",
        "population": c["population"],
        "area_km2": c.get("area", 0)
    })

# Create a DataFrame
df_exercise = pd.DataFrame(rows)
print(f"DataFrame: {len(df_exercise)} rows, {len(df_exercise.columns)} columns")
df_exercise.head()

---
## 5.5 Filtering and selecting

Pandas makes it easy to slice your data.

In [ ]:
# First, let's get a fresh DataFrame with more data
response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,capital,population,area"}
)
countries = response.json()

rows = []
for c in countries:
    rows.append({
        "name": c["name"]["common"],
        "capital": c["capital"][0] if c.get("capital") else "N/A",
        "population": c["population"],
        "area_km2": c.get("area", 0)
    })

df = pd.DataFrame(rows)
print(f"Full table: {len(df)} rows, {len(df.columns)} columns")
df.head()

In [ ]:
# Select specific columns
df_slim = df[["name", "population"]]
df_slim.head()

In [ ]:
# Filter rows: countries with population over 10 million
big_countries = df[df["population"] > 10000000]
print(f"{len(big_countries)} countries have more than 10 million people")
big_countries

In [ ]:
# Sort by population (largest first)
df_sorted = df.sort_values("population", ascending=False)
df_sorted.head(10)

In [ ]:
# Combine: filter + select + sort
result = df[df["population"] > 10000000][["name", "population"]].sort_values("population", ascending=False)
result

**Exercise:** Using the `df` from the cell above, find all European countries with an area over 100,000 km², sorted by area (largest first).

Required variables:
- `large_countries` — a DataFrame of just those countries, sorted by `area_km2` descending
- `large_country_count` — an integer: how many countries that is

In [ ]:
# Find all European countries with area over 100,000 km², sorted by area (largest first)
large_countries = df[df["area_km2"] > 100000].sort_values("area_km2", ascending=False)
large_country_count = len(large_countries)

print(f"{large_country_count} European countries have an area over 100,000 km²\n")
large_countries

---
## 5.6 Exporting to CSV

CSV (Comma-Separated Values) files can be opened in Excel, Google Sheets, or any data tool.

In [ ]:
df_sorted = df.sort_values("population", ascending=False)
df_sorted.to_csv("european_countries.csv", index=False)
print("Saved to european_countries.csv")

The file appears in your project folder (where you launched Jupyter). `index=False` prevents pandas from adding a numbered index column.

---
## 5.7 Exporting to Excel

In [ ]:
df_sorted.to_excel("european_countries.xlsx", index=False)
print("Saved to european_countries.xlsx")

---
## 5.8 Saving raw JSON

Sometimes you want to keep the original API response for reference:

In [ ]:
import json

with open("european_countries_raw.json", "w") as f:
    json.dump(countries, f, indent=2)

print("Saved raw API response to european_countries_raw.json")

---
## 5.9 Organizing outputs

Create an output folder to keep things tidy:

In [ ]:
import os

os.makedirs("output", exist_ok=True)  # Creates the folder if it doesn't exist

df_sorted.to_csv("output/european_countries.csv", index=False)
df_sorted.to_excel("output/european_countries.xlsx", index=False)
print("Saved to output/ folder")

---
## 5.10 Complete example: start to finish

Here's the full pattern you'll use over and over. This is the same thing the sneak peek did in Notebook 0 — but now you understand every line.

In [ ]:
import requests
import pandas as pd
import os

# 1. Call the API
response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,capital,population,area"}
)

if not response.ok:
    print(f"Error: {response.status_code}")
else:
    countries = response.json()

    # 2. Process into clean rows
    rows = []
    for c in countries:
        rows.append({
            "name": c["name"]["common"],
            "capital": c["capital"][0] if c.get("capital") else "N/A",
            "population": c["population"],
            "area_km2": c.get("area", 0)
        })

    # 3. Create DataFrame and sort
    df = pd.DataFrame(rows)
    df = df.sort_values("population", ascending=False)

    # 4. Save
    os.makedirs("output", exist_ok=True)
    df.to_csv("output/european_countries.csv", index=False)

    # 5. Show results
    print(f"Saved {len(df)} countries to output/european_countries.csv")
    print()
    print(df.head(10).to_string(index=False))

---
## 5.11 Checkpoint

Run this cell to verify your understanding.

In [ ]:
# Run this cell to check your understanding.
# It checks the variables you produced in the exercises above.

import pandas as pd

required_vars = [
    "df_exercise",
    "large_countries",
    "large_country_count",
]

missing = [v for v in required_vars if v not in globals() or globals()[v] is None]
assert not missing, f"Missing or unfinished: {missing}. Go back and complete those exercises."

# 5.4 - df_exercise: flattened European countries with 4 columns
assert isinstance(df_exercise, pd.DataFrame), (
    f"df_exercise should be a DataFrame, got {type(df_exercise).__name__}"
)
assert len(df_exercise) == 53, (
    f"df_exercise should have all 53 European countries, got {len(df_exercise)}. "
    "Did your loop process every item?"
)
for col in ["name", "capital", "population", "area_km2"]:
    assert col in df_exercise.columns, f"df_exercise is missing the '{col}' column"

# 5.5 - large_countries: area > 100,000 km², sorted by area descending
assert isinstance(large_countries, pd.DataFrame), (
    f"large_countries should be a DataFrame, got {type(large_countries).__name__}"
)
assert (large_countries["area_km2"] > 100000).all(), (
    "Every row in large_countries should have area_km2 > 100000 — check your filter"
)
areas = large_countries["area_km2"].tolist()
assert areas == sorted(areas, reverse=True), (
    "large_countries should be sorted by area_km2, largest first"
)
assert len(large_countries) == 16, (
    f"Expected 16 countries over 100,000 km², got {len(large_countries)}"
)

# 5.5 - large_country_count: the count
assert large_country_count == 16, (
    f"large_country_count should be 16, got {large_country_count}. "
    "Hint: len(large_countries)"
)

notebook5_ready_for_clue = True
print("All checks passed — you're ready for the clue.")

---
## 5.12 Data Detective Challenge — Clue 5

Put all your skills together: fetch data, process it, sort it, and find a specific value.

In [ ]:
# CHALLENGE: What is the 5th largest European country by population?
#
# Steps:
# 1. Fetch European countries with name and population
# 2. Build a clean list with a loop
# 3. Create a DataFrame
# 4. Sort by population (descending)
# 5. Get the 5th country's name (hint: index 4)

assert notebook5_ready_for_clue, "Run the 5.11 checkpoint first."

import requests
import pandas as pd

response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,population"}
)
data = response.json()

rows = []
for c in data:
    rows.append({
        "name": c["name"]["common"],
        "population": c["population"]
    })

df = pd.DataFrame(rows)
df = df.sort_values("population", ascending=False).reset_index(drop=True)

clue_5 = df.iloc[4]["name"]  # The name of the 5th country (index 4)

print(f"Your answer: {clue_5}")
print(f"\nTop 6 for reference:")
print(df.head(6).to_string(index=False))

# --- Verification ---
valid = ["Italy"]
if clue_5 in valid:
    print(f"\n>>> Clue 5 collected!")
    print(f'Write down: clue_5 = "{clue_5}"')
else:
    print("Not quite — remember index 4 is the 5th item (counting starts at 0).")
    print("Use: df.iloc[4]['name']")

---
**Next up:** [Notebook 6 — Errors, Troubleshooting & Final Challenge](6_errors_troubleshooting_and_finale.ipynb)